# Drought event detection from SPEI-3 (3D patch → link → event workflow)

This notebook implements, function by function, the methodology in the slides
*"Drought Methodology Explained – Input / Output"*.

**Input:** a NetCDF file with monthly SPEI-3 on a regular lat/lon grid (ERA5-derived drought indices, e.g. 0.25°).

**Workflow**

| Step | What happens | Functions |
|---|---|---|
| 0 | read / subset the NetCDF, optional normality QC, true cell areas (km²) | `read_spei_nc`, `subset_spei`, `normality_qc_mask`, `cell_area_km2` |
| 1 | per month: flag cells with SPEI < −1, group touching cells (8 neighbours) into patches, keep patches > 5 % of the basin | `flag_drought`, `find_patches` |
| 2 | link patches of consecutive months if their overlap is > 5 % of the basin (each pair judged alone) | `link_patches` |
| 3 | connected components of the link graph = 3D drought events (union-find); event ids only after **all** links are known | `assign_events`, `build_label_cube` |
| 4 | monthly track + event numbers: duration D, footprint A, area×time V, severity S, intensity I, 3D centroid | `event_tables`, `summarise_events` |
| 5 | outputs O1–O5 (CSV, NetCDF, PNG, JSON) | `write_outputs`, `plot_*` |
| all | everything in one call | **`run_workflow(...)`** |

**Outputs per run:** O1 event catalogue (CSV) · O2 monthly track (CSV) · O3 label cube (NetCDF, int32) · O4 event maps (PNG, one per event) · O5 run log (JSON), plus a few extra figures and the patch/link tables.

**Layout of this notebook:** 1) setup → 2) functions (Sections A–J) → 3) test on the toy basin from the slides → 4) test on generated dummy NetCDF data → 5) how to run on your own data.

## 1. Setup

In [ ]:
import os, io, json, math, sys, platform, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch as LegendPatch
from matplotlib.lines import Line2D
from scipy import ndimage, stats

try:                       # xarray is pre-installed on Google Colab
    import xarray as xr
except ImportError:        # NetCDF3 fallback through scipy (works, but xarray is recommended)
    xr = None

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 9,
    "axes.titlesize": 10, "axes.spines.top": False, "axes.spines.right": False,
})

R_EARTH_KM = 6371.0088
STRUCT8 = np.ones((3, 3), dtype=int)                      # 8-neighbour connectivity (sides + corners)
STRUCT4 = ndimage.generate_binary_structure(2, 1)         # 4-neighbour (optional)

# Drought classes (display only, no gaps):  Moderate -1.5 < SPEI < -1 | Severe -2 < SPEI <= -1.5 | Extreme SPEI <= -2
CLASS_COLORS = ["#fdae6b", "#e6550d", "#7f2704"]
CLASS_LABELS = ["Moderate (−1.5 < SPEI < −1)", "Severe (−2 < SPEI ≤ −1.5)", "Extreme (SPEI ≤ −2)"]
CLASS_CMAP = ListedColormap(CLASS_COLORS)
LAND_COLOR = "#e6e6e6"
PATCH_COLORS = [c for i, c in enumerate(plt.get_cmap("tab10").colors) if i != 7]   # categorical colours, no grey


def _pcolor(i):
    return PATCH_COLORS[int(i) % len(PATCH_COLORS)]
print("xarray available:", xr is not None, "| numpy", np.__version__, "| pandas", pd.__version__)

## 2. Functions
### A. Small helpers

In [ ]:
def patch_name(i):
    """0 -> 'A', 1 -> 'B', ... 25 -> 'Z', 26 -> 'AA' ... (patches are named in time order)."""
    s, i = "", int(i) + 1
    while i > 0:
        i, r = divmod(i - 1, 26)
        s = chr(65 + r) + s
    return s


def cell_area_km2(lat, lon):
    """True area (km²) of every grid cell, shape (n_lat, n_lon). Cells shrink towards the poles:
    A = R² · Δlon · |sin(lat+Δlat/2) − sin(lat−Δlat/2)|"""
    lat = np.asarray(lat, float)
    lon = np.asarray(lon, float)
    dlat = abs(float(np.median(np.diff(lat)))) if lat.size > 1 else 0.25
    dlon = abs(float(np.median(np.diff(lon)))) if lon.size > 1 else 0.25
    lat1, lat2 = np.radians(lat - dlat / 2), np.radians(lat + dlat / 2)
    band = R_EARTH_KM ** 2 * np.radians(dlon) * np.abs(np.sin(lat2) - np.sin(lat1))
    return np.repeat(band[:, None], lon.size, axis=1)


def _edges(c):
    """Cell-centre coordinates -> cell edges (for pcolormesh)."""
    c = np.asarray(c, float)
    if c.size == 1:
        return np.array([c[0] - 0.125, c[0] + 0.125])
    mid = (c[1:] + c[:-1]) / 2
    return np.concatenate([[c[0] - (mid[0] - c[0])], mid, [c[-1] + (c[-1] - mid[-1])]])


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, (pd.Timestamp, np.datetime64)):
        return str(pd.Timestamp(o).date())
    return str(o)


def _finish(fig, path=None, show=True, dpi=150):
    """Save a figure (if path given), then show it in the notebook or close it."""
    if path is not None:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return path


def show_table(df, title=None, max_rows=15):
    """Display a DataFrame (first max_rows rows) with an optional title."""
    if title:
        print(f"\n{title}  [{len(df)} rows]")
    display(df.head(max_rows))

### B. NetCDF reading, subsetting and writing
`read_spei_nc` accepts any NetCDF with a time, a latitude and a longitude coordinate and one SPEI variable
(names such as `time/valid_time`, `latitude/lat`, `longitude/lon`, `spei3/spei_3/spei`). The result is always sorted
south→north, west→east, longitudes are converted to −180…180 and times to the first day of the month.

In [ ]:
def _pick(names, cands):
    low = {str(n).lower(): n for n in names}
    for c in cands:
        if c in low:
            return low[c]
    return None


def read_spei_nc(path, var=None):
    """Read monthly SPEI from NetCDF. Returns dict(spei[T,Y,X], lat, lon, time, var, attrs)."""
    if xr is not None:
        ds = xr.open_dataset(path)
        tname = _pick(list(ds.coords) + list(ds.dims), ["time", "valid_time", "date"])
        yname = _pick(list(ds.coords) + list(ds.dims), ["latitude", "lat", "y"])
        xname = _pick(list(ds.coords) + list(ds.dims), ["longitude", "lon", "x"])
        if None in (tname, yname, xname):
            raise ValueError(f"Could not find time/lat/lon coordinates in {list(ds.coords)}")
        if var is None:
            cands = [v for v in ds.data_vars if str(v).lower().startswith("spei")]
            cands = cands or [v for v in ds.data_vars if ds[v].ndim >= 3]
            if not cands:
                raise ValueError("No SPEI variable found; pass var='name'.")
            var = cands[0]
        da = ds[var]
        for d in list(da.dims):
            if d not in (tname, yname, xname):
                if da.sizes[d] == 1:
                    da = da.isel({d: 0})
                else:
                    raise ValueError(f"Extra dimension '{d}' of size {da.sizes[d]} - select one slice first.")
        da = da.transpose(tname, yname, xname)
        spei = da.values.astype("float64")
        lat, lon = ds[yname].values.astype(float), ds[xname].values.astype(float)
        time = pd.DatetimeIndex(ds[tname].values)
        attrs = dict(da.attrs)
        ds.close()
    else:
        from scipy.io import netcdf_file
        f = netcdf_file(path, "r", mmap=False)
        names = list(f.variables)
        tname = _pick(names, ["time", "valid_time", "date"])
        yname = _pick(names, ["latitude", "lat", "y"])
        xname = _pick(names, ["longitude", "lon", "x"])
        if var is None:
            cands = [v for v in names if v.lower().startswith("spei")] or \
                    [v for v in names if len(f.variables[v].dimensions) >= 3]
            var = cands[0]
        v = f.variables[var]
        data = np.array(v[:], dtype="float64")
        dims = list(v.dimensions)
        data = np.transpose(data, [dims.index(tname), dims.index(yname), dims.index(xname)])
        fill = getattr(v, "_FillValue", None)
        if fill is not None:
            data[data == float(np.asarray(fill).ravel()[0])] = np.nan
        spei = data
        lat, lon = np.array(f.variables[yname][:], float), np.array(f.variables[xname][:], float)
        tv = f.variables[tname]
        units = tv.units.decode() if isinstance(tv.units, bytes) else str(tv.units)
        unit, _, base = units.partition(" since ")
        step = {"days": "D", "day": "D", "hours": "h", "hour": "h"}[unit.strip().lower()]
        time = pd.DatetimeIndex(pd.Timestamp(base.strip()) + pd.to_timedelta(np.array(tv[:], float), unit=step))
        attrs = {}
        f.close()
    spei[np.abs(spei) > 1e10] = np.nan
    lon = ((lon + 180.0) % 360.0) - 180.0                       # -180 ... 180
    o_lat, o_lon = np.argsort(lat), np.argsort(lon)             # south->north, west->east
    spei, lat, lon = spei[:, o_lat, :][:, :, o_lon], lat[o_lat], lon[o_lon]
    time = time.to_period("M").to_timestamp()                   # first day of month
    order = np.argsort(time.values)
    return dict(spei=spei[order], lat=lat, lon=lon, time=time[order], var=var, attrs=attrs, path=path)


def subset_spei(data, bbox=None, period=None):
    """Clip to a study area and/or period.
    bbox   = dict(north=50, south=45, west=-1, east=6)   (None = whole grid)
    period = ('1980-01', '2020-12')                       (None = whole record, both ends inclusive)"""
    spei, lat, lon, time = data["spei"], data["lat"], data["lon"], data["time"]
    if bbox:
        eps = 1e-6
        iy = (lat >= bbox["south"] - eps) & (lat <= bbox["north"] + eps)
        ix = (lon >= bbox["west"] - eps) & (lon <= bbox["east"] + eps)
        if not iy.any() or not ix.any():
            raise ValueError("bbox does not overlap the data grid.")
        spei, lat, lon = spei[:, iy, :][:, :, ix], lat[iy], lon[ix]
    if period:
        per = time.to_period("M")
        it = (per >= pd.Period(period[0], "M")) & (per <= pd.Period(period[1], "M"))
        if not it.any():
            raise ValueError("period does not overlap the data.")
        spei, time = spei[it], time[it]
    out = dict(data)
    out.update(spei=spei, lat=lat, lon=lon, time=time)
    steps = np.diff(time.to_period("M").astype("int64"))
    if (steps != 1).any():
        warnings.warn("Time axis has gaps or duplicates; months are linked by position, not by date.")
    return out


def write_nc(path, var, data, lat, lon, times, dtype="float32", attrs=None,
             lat_name="latitude", lon_name="longitude"):
    """Write a (time, lat, lon) array to NetCDF (xarray if available, otherwise scipy NetCDF3)."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    attrs = {k: str(v) for k, v in (attrs or {}).items()}
    is_int = np.issubdtype(np.dtype(dtype), np.integer)
    times = pd.DatetimeIndex(times)
    if xr is not None:
        ds = xr.Dataset(
            {var: (("time", lat_name, lon_name), np.asarray(data).astype(dtype), attrs)},
            coords={"time": times, lat_name: np.asarray(lat, float), lon_name: np.asarray(lon, float)})
        ds[lat_name].attrs.update(units="degrees_north", long_name="latitude")
        ds[lon_name].attrs.update(units="degrees_east", long_name="longitude")
        enc = {var: {"dtype": dtype}}
        if is_int:
            enc[var]["_FillValue"] = None
        ds.to_netcdf(path, encoding=enc)
    else:
        from scipy.io import netcdf_file
        f = netcdf_file(path, "w")
        f.createDimension("time", len(times))
        f.createDimension(lat_name, len(lat))
        f.createDimension(lon_name, len(lon))
        tv = f.createVariable("time", "i4", ("time",))
        tv[:] = ((times - pd.Timestamp("1970-01-01")).days).values.astype("int32")
        tv.units, tv.calendar = "days since 1970-01-01 00:00:00", "standard"
        yv = f.createVariable(lat_name, "f8", (lat_name,)); yv[:] = np.asarray(lat, float); yv.units = "degrees_north"
        xv = f.createVariable(lon_name, "f8", (lon_name,)); xv[:] = np.asarray(lon, float); xv.units = "degrees_east"
        dv = f.createVariable(var, "i4" if is_int else "f4", ("time", lat_name, lon_name))
        dv[:] = np.asarray(data).astype("i4" if is_int else "f4")
        for k, v in attrs.items():
            setattr(dv, k, v)
        f.close()
    return path

### C. Dummy-data generator
Builds a realistic test file on the sample grid of the slides (N 50, S 45, W 1°W, E 6°E, 0.25° → 21 × 29 cells,
monthly 1980–2023): spatially smooth, month-to-month persistent noise standardised per cell (≈ SPEI-3 behaviour), a
block of NaN "sea" cells, and several planted drought events that move, grow, shrink and sometimes split.

In [ ]:
def make_dummy_spei(out_path, north=50, south=45, west=-1, east=6, start="1980-01", end="2023-12",
                    res=0.25, n_events=14, seed=42, var="spei3", sea=True):
    """Create a dummy SPEI-3 NetCDF file for testing. Returns the path."""
    rng = np.random.default_rng(seed)
    lat = np.arange(south, north + res / 2, res)
    lon = np.arange(west, east + res / 2, res)
    times = pd.date_range(start, end, freq="MS")
    T, Y, X = len(times), len(lat), len(lon)

    # 1) background: smooth in space, AR(1) in time, standardised per cell
    noise = ndimage.gaussian_filter(rng.standard_normal((T, Y, X)), sigma=(0, 1.0, 1.0), mode="nearest")
    f = np.zeros_like(noise)
    phi = 0.65
    f[0] = noise[0]
    for t in range(1, T):
        f[t] = phi * f[t - 1] + np.sqrt(1 - phi ** 2) * noise[t]
    f = (f - f.mean(0)) / f.std(0)

    # 2) planted droughts (moving Gaussian blobs, some with a second lobe -> split / merge)
    yy, xx = np.meshgrid(lat, lon, indexing="ij")
    for _ in range(n_events):
        t0 = int(rng.integers(3, T - 14))
        dur = int(rng.integers(3, 10))
        cy = rng.uniform(lat.min() + 1.0, lat.max() - 1.0)
        cx = rng.uniform(lon.min() + 1.2, lon.max() - 1.2)
        vy, vx = rng.normal(0, 0.12, 2)
        r, amp = rng.uniform(0.9, 1.6), rng.uniform(1.6, 2.8)
        lobe = rng.random() < 0.4
        dy, dx = rng.choice([-1, 1]) * 1.8, rng.choice([-1, 1]) * 1.8
        for m in range(dur):
            env = np.sin(np.pi * (m + 0.5) / dur) ** 0.7
            rm = r * (0.8 + 0.4 * env)
            gy, gx = cy + vy * m, cx + vx * m
            f[t0 + m] -= amp * env * np.exp(-((yy - gy) ** 2 + (xx - gx) ** 2) / (2 * rm ** 2))
            if lobe and m >= dur // 2:
                f[t0 + m] -= 0.8 * amp * env * np.exp(-((yy - gy - dy) ** 2 + (xx - gx - dx) ** 2) / (2 * (0.7 * rm) ** 2))
    f = np.clip(f, -3.5, 3.5)

    # 3) NaN "sea" block in the north-west corner (like the Channel / Atlantic)
    if sea:
        f[:, (yy >= 48.75) & (xx <= 0.75)] = np.nan

    write_nc(out_path, var, f, lat, lon, times, dtype="float32",
             attrs=dict(long_name="Standardised Precipitation-Evapotranspiration Index (accumulation 3 months)",
                        units="1", note="DUMMY DATA for testing the workflow"))
    return out_path

### D. Step 0 – quality control (optional) and study-area bookkeeping

In [ ]:
def normality_qc_mask(spei, alpha=0.05, min_n=8):
    """Shapiro-Wilk normality test of each cell's SPEI series (optional quality criterion, alpha = 0.05).
    Returns (ok[Y,X] bool, pvalue[Y,X]); ok = True where normality is NOT rejected."""
    T, Y, X = spei.shape
    ok = np.zeros((Y, X), bool)
    pv = np.full((Y, X), np.nan)
    for y in range(Y):
        for x in range(X):
            s = spei[:, y, x]
            s = s[np.isfinite(s)]
            if s.size >= min_n and np.ptp(s) > 0:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    pv[y, x] = stats.shapiro(s).pvalue
                ok[y, x] = pv[y, x] >= alpha
    return ok, pv


def study_area_stats(spei, area):
    """Total basin area A_tot = area of all cells that have data; plus the share of no-data (e.g. sea) cells."""
    valid = np.isfinite(spei).any(axis=0)
    return dict(A_tot=float(area[valid].sum()), n_cells_total=int(valid.size), n_cells_valid=int(valid.sum()),
                share_no_data=float(1 - valid.mean()), valid_mask=valid)

### E. Step 1 – drought cells and patches in every month
* drought cell: **SPEI < threshold** (strictly; −1.0 does not count)
* patch: touching drought cells (8 neighbours) in the same month
* keep a patch only if **area > min_area_frac × A_tot** (5 %); small patches and single cells are dropped for good
* patches get global ids and names A, B, C … in time order

In [ ]:
def flag_drought(spei_t, threshold=-1.0):
    """Boolean map of drought cells of one month (NaN is never drought)."""
    return np.isfinite(spei_t) & (spei_t < threshold)


def find_patches(spei, area, threshold=-1.0, min_patch_area=0.0, connectivity=8):
    """Step 1 for all months.
    Returns
      patch_cube [T,Y,X] int32 : global patch id (0 = none / dropped)
      patch_df   : one row per kept patch (patch_id, name, t_index (1-based), n_cells, area_km2)
      month_df   : one row per month (drought cells, kept / dropped patches, areas)"""
    struct = STRUCT8 if connectivity == 8 else STRUCT4
    T = spei.shape[0]
    cube = np.zeros(spei.shape, np.int32)
    rows, mrows, nxt = [], [], 0
    for t in range(T):
        dmask = flag_drought(spei[t], threshold)
        lab, n = ndimage.label(dmask, structure=struct)
        rec = dict(t_index=t + 1, n_drought_cells=int(dmask.sum()), area_drought_km2=float(area[dmask].sum()),
                   n_patches_all=int(n), n_kept=0, n_dropped_small=0, n_dropped_single=0, area_kept_km2=0.0)
        if n > 0:
            idx = np.arange(1, n + 1)
            a = np.asarray(ndimage.sum(area, lab, idx))
            c = np.asarray(ndimage.sum(dmask, lab, idx))
            keep = a > min_patch_area                               # strictly larger than 5 % of the basin
            remap = np.zeros(n + 1, np.int32)
            ids = np.arange(nxt + 1, nxt + 1 + int(keep.sum()))
            remap[1:][keep] = ids
            cube[t] = remap[lab]
            for pid, k in zip(ids, np.nonzero(keep)[0]):
                rows.append(dict(patch_id=int(pid), name=patch_name(pid - 1), t_index=t + 1,
                                 n_cells=int(c[k]), area_km2=float(a[k])))
            nxt += int(keep.sum())
            rec.update(n_kept=int(keep.sum()), n_dropped_single=int(((~keep) & (c == 1)).sum()),
                       n_dropped_small=int(((~keep) & (c > 1)).sum()), area_kept_km2=float(a[keep].sum()))
        mrows.append(rec)
    patch_df = pd.DataFrame(rows, columns=["patch_id", "name", "t_index", "n_cells", "area_km2"])
    month_df = pd.DataFrame(mrows)
    return cube, patch_df, month_df

### F. Step 2 – link patches in consecutive months
For every pair (p in month t, q in month t+1): overlap O(p,q) = shared area. **Linked if O > link_frac × A_tot (5 %)**.
Each pair is judged on its own – overlaps are never added together.

In [ ]:
def link_patches(patch_cube, area, link_min_area, patch_df):
    """Step 2. Returns a DataFrame with every overlapping pair of patches in consecutive months and a `linked` flag."""
    T = patch_cube.shape[0]
    M = int(patch_cube.max()) + 1
    names = patch_df.set_index("patch_id")["name"].to_dict()
    rows = []
    for t in range(T - 1):
        a, b = patch_cube[t], patch_cube[t + 1]
        m = (a > 0) & (b > 0)
        if not m.any():
            continue
        key = a[m].astype(np.int64) * M + b[m].astype(np.int64)
        uk, inv = np.unique(key, return_inverse=True)
        inv = inv.ravel()
        ov_area = np.bincount(inv, weights=area[m])
        ov_cells = np.bincount(inv)
        for k, oa, oc in zip(uk, ov_area, ov_cells):
            p, q = divmod(int(k), M)
            rows.append(dict(patch_p=p, patch_q=q, name_p=names[p], name_q=names[q], t_index_p=t + 1,
                             overlap_cells=int(oc), overlap_km2=float(oa), linked=bool(oa > link_min_area)))
    cols = ["patch_p", "patch_q", "name_p", "name_q", "t_index_p", "overlap_cells", "overlap_km2", "linked"]
    return pd.DataFrame(rows, columns=cols)

### G. Step 3 – chain links into 3D events
Patches are nodes, passing links are edges; every connected group is one event (union-find). Splits and merges stay in
one event. Event ids are given **after all links are known** (a patch without a link to the previous month starts a new
event only if it is not later merged into an older one). Ids follow the time of the first patch.

In [ ]:
def assign_events(n_patches, links_df):
    """Union-find over the passing links. Returns event_of_patch (length n_patches+1; index 0 = background 0)."""
    parent = np.arange(n_patches + 1)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    if len(links_df):
        for p, q in links_df.loc[links_df["linked"], ["patch_p", "patch_q"]].to_numpy():
            rp, rq = find(int(p)), find(int(q))
            if rp != rq:
                parent[max(rp, rq)] = min(rp, rq)
    event_of_patch = np.zeros(n_patches + 1, np.int32)
    root_to_event, nxt = {}, 0
    for pid in range(1, n_patches + 1):                      # patch ids are in time order
        r = find(pid)
        if r not in root_to_event:
            nxt += 1
            root_to_event[r] = nxt
        event_of_patch[pid] = root_to_event[r]
    return event_of_patch


def build_label_cube(patch_cube, event_of_patch):
    """event_id[time, lat, lon] (int32, 0 = no event)."""
    return event_of_patch[patch_cube].astype(np.int32)

### H. Event characteristics
* **monthly track** (O2): per event-month – patches, area, severity Σ|SPEI|·A, mean |SPEI|, centre (lon, lat) weighted by |SPEI|·A
* **duration** D = last − first + 1 · **footprint** A = cells hit at least once · **area × time** V = Σ monthly areas
* **severity** S = Σ|SPEI|·A · **intensity** I = S / V
* **3D centroid** = weighted average of the monthly centres, weights w = monthly Σ|SPEI|·A, t = month number (1 = first month of the record)

In [ ]:
def tc_to_date(tc, times):
    """Fractional month number (1 = first month of the record) -> calendar date (e.g. 1.83 -> 26 Jan)."""
    ms = pd.DatetimeIndex(times).to_period("M").to_timestamp()
    i = int(np.clip(np.floor(tc) - 1, 0, len(ms) - 1))
    frac = tc - np.floor(tc)
    return (ms[i] + pd.Timedelta(days=float(frac) * ms[i].days_in_month)).normalize()


def event_tables(spei, label_cube, patch_cube, patch_df, area, lon, lat, times):
    """Monthly track (O2), footprint area and patch count for every event."""
    t, y, x = np.nonzero(label_cube)
    cols = ["event_id", "month", "patch_ids", "area_km2", "sev", "mean_abs_spei", "lon_c", "lat_c", "n_cells", "t_index"]
    if t.size == 0:
        return pd.DataFrame(columns=cols), pd.Series(dtype=float), pd.Series(dtype=int)
    w = np.abs(spei[t, y, x]) * area[y, x]
    df = pd.DataFrame(dict(event_id=label_cube[t, y, x], t=t, y=y, x=x, a=area[y, x], w=w,
                           wx=w * np.asarray(lon)[x], wy=w * np.asarray(lat)[y], pid=patch_cube[t, y, x]))
    g = df.groupby(["event_id", "t"])
    track = g.agg(area_km2=("a", "sum"), sev=("w", "sum"), wx=("wx", "sum"), wy=("wy", "sum"), n_cells=("a", "size"))
    names = patch_df.set_index("patch_id")["name"]
    track["patch_ids"] = g["pid"].agg(lambda s: "+".join(names.loc[sorted(set(s))]))
    track = track.reset_index()
    track["mean_abs_spei"] = track["sev"] / track["area_km2"]
    track["lon_c"] = track["wx"] / track["sev"]
    track["lat_c"] = track["wy"] / track["sev"]
    track["t_index"] = track["t"] + 1
    track["month"] = pd.DatetimeIndex(times)[track["t"].values].strftime("%Y-%m")
    track = track[cols].sort_values(["event_id", "t_index"]).reset_index(drop=True)
    foot = df.drop_duplicates(["event_id", "y", "x"]).groupby("event_id")["a"].sum()
    npatch = df.groupby("event_id")["pid"].nunique()
    return track, foot, npatch


def summarise_events(track, footprint_area, n_patches, A_tot, times):
    """Event catalogue (O1), one row per event, from the monthly track."""
    cols = ["event_id", "t_start", "t_end", "D", "A_km2", "A_pct", "V", "S", "I", "lon_c", "lat_c", "t_c",
            "t_c_date", "n_patches", "peak_month", "dur_ge3", "censored"]
    if len(track) == 0:
        return pd.DataFrame(columns=cols)
    T = len(times)
    rows = []
    for e, g in track.groupby("event_id"):
        g = g.sort_values("t_index")
        S = g["sev"].sum()
        V = g["area_km2"].sum()
        tc = (g["sev"] * g["t_index"]).sum() / S
        t0, t1 = int(g["t_index"].min()), int(g["t_index"].max())
        rows.append(dict(
            event_id=int(e), t_start=g["month"].iloc[0], t_end=g["month"].iloc[-1], D=t1 - t0 + 1,
            A_km2=float(footprint_area.loc[e]), A_pct=100 * float(footprint_area.loc[e]) / A_tot,
            V=V, S=S, I=S / V,
            lon_c=(g["sev"] * g["lon_c"]).sum() / S, lat_c=(g["sev"] * g["lat_c"]).sum() / S, t_c=tc,
            t_c_date=tc_to_date(tc, times).strftime("%Y-%m-%d"),
            n_patches=int(n_patches.loc[e]), peak_month=g.loc[g["sev"].idxmax(), "month"],
            dur_ge3=bool(t1 - t0 + 1 >= 3), censored=bool(t0 == 1 or t1 == T)))
    return pd.DataFrame(rows, columns=cols)


def detect_events(spei, lat, lon, times, area, threshold=-1.0, min_area_frac=0.05, link_frac=0.05,
                  connectivity=8, A_tot=None):
    """Core engine: Steps 1-3 + event numbers on arrays. Returns a results dict used by plots and writers."""
    if A_tot is None:
        A_tot = study_area_stats(spei, area)["A_tot"]
    min_patch_area, link_min_area = min_area_frac * A_tot, link_frac * A_tot
    patch_cube, patch_df, month_df = find_patches(spei, area, threshold, min_patch_area, connectivity)
    month_df["month"] = pd.DatetimeIndex(times)[month_df["t_index"].values - 1].strftime("%Y-%m")
    patch_df["month"] = pd.DatetimeIndex(times)[patch_df["t_index"].values - 1].strftime("%Y-%m")
    links_df = link_patches(patch_cube, area, link_min_area, patch_df)
    event_of_patch = assign_events(len(patch_df), links_df)
    label_cube = build_label_cube(patch_cube, event_of_patch)
    patch_df["event_id"] = event_of_patch[patch_df["patch_id"].values]
    track, foot, npatch = event_tables(spei, label_cube, patch_cube, patch_df, area, lon, lat, times)
    catalogue = summarise_events(track, foot, npatch, A_tot, times)
    return dict(patch_cube=patch_cube, patch_df=patch_df, month_df=month_df, links_df=links_df,
                event_of_patch=event_of_patch, label_cube=label_cube, track=track, catalogue=catalogue,
                A_tot=A_tot, min_patch_area=min_patch_area, link_min_area=link_min_area,
                threshold=threshold, connectivity=connectivity)

### I. Visualisation functions
Every function returns the figure; `_finish` saves and shows it. Maps use plain lon/lat axes (no extra libraries needed).

In [ ]:
def classify_spei(s):
    """Drought class code for display: 1 moderate, 2 severe, 3 extreme (0 = none)."""
    return np.where(s <= -2, 3, np.where(s <= -1.5, 2, np.where(np.isfinite(s), 1, 0))).astype(np.int8)


def _base_map(ax, lon_e, lat_e, valid, aspect):
    ax.pcolormesh(lon_e, lat_e, np.ma.masked_where(~valid, np.ones(valid.shape)),
                  cmap=ListedColormap([LAND_COLOR]), shading="flat")
    ax.plot([lon_e[0], lon_e[-1], lon_e[-1], lon_e[0], lon_e[0]],
            [lat_e[0], lat_e[0], lat_e[-1], lat_e[-1], lat_e[0]], color="k", lw=0.8)   # study-area outline
    ax.set_xlim(lon_e[0], lon_e[-1]); ax.set_ylim(lat_e[0], lat_e[-1]); ax.set_aspect(aspect)
    ax.tick_params(labelsize=7)


def _aspect(res, lat):
    return 1.0 if res.get("unit_cell_area") else 1.0 / max(np.cos(np.radians(np.mean(lat))), 0.2)


def plot_overview(res, spei, lat, lon, times, threshold=-1.0):
    """Input overview: mean SPEI map, basin-mean SPEI series, share of basin in drought per month."""
    fig = plt.figure(figsize=(13, 3.6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.5, 1.5])
    lon_e, lat_e = _edges(lon), _edges(lat)
    ax = fig.add_subplot(gs[0])
    m = ax.pcolormesh(lon_e, lat_e, np.nanmean(spei, axis=0), cmap="RdBu", vmin=-1, vmax=1)
    ax.set_aspect(_aspect(res, lat)); ax.set_title("Mean SPEI per cell"); fig.colorbar(m, ax=ax, shrink=0.8)
    ax2 = fig.add_subplot(gs[1])
    ser = np.nanmean(spei, axis=(1, 2))
    ax2.fill_between(times, ser, 0, where=ser < 0, color="#e6550d", alpha=0.8, lw=0)
    ax2.fill_between(times, ser, 0, where=ser >= 0, color="#6baed6", alpha=0.8, lw=0)
    ax2.axhline(threshold, color="k", lw=0.7, ls="--"); ax2.set_title("Basin-mean SPEI"); ax2.set_ylabel("SPEI")
    ax3 = fig.add_subplot(gs[2])
    frac = 100 * res["month_df"]["area_drought_km2"] / res["A_tot"]
    ax3.bar(times, frac, width=25, color="#e6550d")
    ax3.axhline(100 * res["min_patch_area"] / res["A_tot"], color="k", lw=0.8, ls="--")
    ax3.set_title(f"% of basin with SPEI < {threshold:g}  (dashed = patch size limit)"); ax3.set_ylabel("% of basin")
    fig.tight_layout()
    return fig


def plot_qc(ok, pvals, lat, lon, alpha=0.05):
    """Normality QC: Shapiro-Wilk p-value map and pass/fail map."""
    lon_e, lat_e = _edges(lon), _edges(lat)
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
    m = axs[0].pcolormesh(lon_e, lat_e, pvals, cmap="viridis", vmin=0, vmax=1)
    fig.colorbar(m, ax=axs[0], shrink=0.8); axs[0].set_title("Shapiro-Wilk p-value")
    cls = np.where(np.isfinite(pvals), np.where(ok, 1, 2), 0)
    axs[1].pcolormesh(lon_e, lat_e, np.ma.masked_equal(cls, 0), cmap=ListedColormap(["#74c476", "#d7301f"]), vmin=0.5, vmax=2.5)
    axs[1].set_title(f"QC (alpha = {alpha}): {100 * ok.sum() / max(np.isfinite(pvals).sum(), 1):.0f}% pass")
    axs[1].legend(handles=[LegendPatch(color="#74c476", label="normal (kept)"), LegendPatch(color="#d7301f", label="rejected (masked)")],
                  fontsize=7, loc="lower left")
    for a in axs:
        a.set_aspect(1.0 / max(np.cos(np.radians(np.mean(lat))), 0.2))
    fig.tight_layout()
    return fig


def plot_step1_patches(res, spei, lat, lon, times, months=None, max_months=4):
    """Step-1 picture: all drought cells of a month; kept patches coloured + named, dropped ones grey."""
    T = spei.shape[0]
    if months is None:
        months = list(range(min(T, max_months)))
    lon_e, lat_e = _edges(lon), _edges(lat)
    valid = np.isfinite(spei).any(0)
    fig, axs = plt.subplots(1, len(months), figsize=(4.0 * len(months), 3.2), squeeze=False)
    for ax, t in zip(axs[0], months):
        _base_map(ax, lon_e, lat_e, valid, _aspect(res, lat))
        d = flag_drought(spei[t], res["threshold"])
        ax.pcolormesh(lon_e, lat_e, np.ma.masked_where(~d, np.ones(d.shape)), cmap=ListedColormap(["#9e9e9e"]), shading="flat")
        pc = res["patch_cube"][t]
        for pid in np.unique(pc[pc > 0]):
            m = pc == pid
            ax.pcolormesh(lon_e, lat_e, np.ma.masked_where(~m, np.ones(m.shape)), cmap=ListedColormap([_pcolor(pid)]), shading="flat")
            yy, xx = np.nonzero(m)
            ax.text(np.mean(np.asarray(lon)[xx]), np.mean(np.asarray(lat)[yy]), res["patch_df"].set_index("patch_id").loc[pid, "name"],
                    ha="center", va="center", fontsize=11, fontweight="bold", color="white",
                    bbox=dict(boxstyle="round,pad=0.15", fc="k", ec="none", alpha=0.55))
        r = res["month_df"].iloc[t]
        ax.set_title(f"{pd.Timestamp(times[t]):%Y-%m}: kept {int(r.n_kept)} | dropped {int(r.n_dropped_small)} small + {int(r.n_dropped_single)} single", fontsize=8)
    fig.legend(handles=[LegendPatch(color="#9e9e9e", label="dropped (too small / single cell)"),
                        LegendPatch(color=_pcolor(0), label="kept patches (> 5 % of basin), coloured and named")],
               loc="lower center", ncol=2, fontsize=8, frameon=False)
    fig.suptitle("Step 1 – drought patches per month", y=1.0)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    return fig


def plot_link_graph(res, times, event_id=None, max_patches=80):
    """Step-2/3 picture: patches = nodes (columns = months), passing links = solid lines, failed overlaps = dashed red."""
    pdf, ldf = res["patch_df"], res["links_df"]
    if event_id is not None:
        pdf = pdf[pdf["event_id"] == event_id]
        ldf = ldf[ldf["patch_p"].isin(pdf["patch_id"]) & ldf["patch_q"].isin(pdf["patch_id"])]
    pdf = pdf.head(max_patches)
    pos = {}
    for t, g in pdf.groupby("t_index"):
        for k, pid in enumerate(g["patch_id"]):
            pos[pid] = (t, -(k - (len(g) - 1) / 2))
    fig, ax = plt.subplots(figsize=(max(5, 0.9 * (pdf["t_index"].nunique() + 1)), 3.6))
    for r in ldf.itertuples():
        if r.patch_p in pos and r.patch_q in pos:
            (x0, y0), (x1, y1) = pos[r.patch_p], pos[r.patch_q]
            ax.plot([x0, x1], [y0, y1], color="k" if r.linked else "#d7301f", lw=1.6 if r.linked else 1.0,
                    ls="-" if r.linked else "--", zorder=1)
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, f"{r.overlap_cells}", fontsize=7, ha="center", va="center",
                    color="k" if r.linked else "#d7301f", bbox=dict(fc="white", ec="none", pad=0.5, alpha=0.9), zorder=3)
    for r in pdf.itertuples():
        x, y = pos[r.patch_id]
        ax.scatter([x], [y], s=520, color=_pcolor(r.event_id), zorder=2, edgecolor="k")
        ax.text(x, y, r.name, ha="center", va="center", fontsize=9, color="white", fontweight="bold", zorder=4)
    ax.set_xticks(sorted(pdf["t_index"].unique()))
    ax.set_xticklabels([f"{pd.Timestamp(times[t - 1]):%Y-%m}" for t in sorted(pdf["t_index"].unique())], rotation=45, fontsize=7)
    ax.set_yticks([]); ax.spines["left"].set_visible(False); ax.margins(0.12)
    ax.set_title("Link graph (node colour = event, label on line = overlap in cells; red dashed = overlap too small)")
    fig.tight_layout()
    return fig


def plot_event_map(res, spei, lat, lon, times, event_id, max_panels=12):
    """O4 – monthly maps of one event (drought cells coloured by class) + footprint coloured by the worst class reached,
    study-area outline and centroid path with arrows."""
    cube, tr = res["label_cube"], res["track"]
    te = tr[tr["event_id"] == event_id].sort_values("t_index")
    ce = res["catalogue"].set_index("event_id").loc[event_id]
    tis = te["t_index"].values - 1
    if len(tis) > max_panels:
        tis_show = tis[np.unique(np.round(np.linspace(0, len(tis) - 1, max_panels)).astype(int))]
    else:
        tis_show = tis
    n = len(tis_show) + 1
    ncols = min(4, n)
    nrows = int(math.ceil(n / ncols))
    lon_e, lat_e = _edges(lon), _edges(lat)
    valid = np.isfinite(spei).any(0)
    asp = _aspect(res, lat)
    ratio = float(np.clip((lat_e[-1] - lat_e[0]) * asp / (lon_e[-1] - lon_e[0]), 0.4, 1.3))
    fig, axs = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.6 * ratio * nrows + 1.0), squeeze=False)
    cls_all = np.zeros(valid.shape, np.int8)
    for t in tis:
        cls_all = np.maximum(cls_all, np.where(cube[t] == event_id, classify_spei(spei[t]), 0))
    flat = axs.ravel()
    for ax, t in zip(flat, tis_show):
        _base_map(ax, lon_e, lat_e, valid, asp)
        cls = np.where(cube[t] == event_id, classify_spei(spei[t]), 0)
        ax.pcolormesh(lon_e, lat_e, np.ma.masked_equal(cls, 0), cmap=CLASS_CMAP, vmin=0.5, vmax=3.5, shading="flat")
        r = te[te["t_index"] == t + 1].iloc[0]
        ax.plot(r.lon_c, r.lat_c, "o", ms=5, mfc="white", mec="k", mew=1)
        ax.set_title(f"{pd.Timestamp(times[t]):%Y-%m}  ·  patches {r.patch_ids}", fontsize=8)
    ax = flat[len(tis_show)]
    _base_map(ax, lon_e, lat_e, valid, asp)
    ax.pcolormesh(lon_e, lat_e, np.ma.masked_equal(cls_all, 0), cmap=CLASS_CMAP, vmin=0.5, vmax=3.5, shading="flat")
    xs, ys = te["lon_c"].values, te["lat_c"].values
    ax.plot(xs, ys, "-", color="k", lw=1, zorder=3)
    for i in range(len(xs) - 1):
        ax.annotate("", xy=(xs[i + 1], ys[i + 1]), xytext=(xs[i], ys[i]), zorder=4,
                    arrowprops=dict(arrowstyle="-|>", color="k", lw=1.2, mutation_scale=10))
    ax.plot(xs, ys, "o", ms=4, mfc="white", mec="k", zorder=5)
    ax.plot(ce.lon_c, ce.lat_c, "*", ms=14, mfc="gold", mec="k", zorder=6)
    ax.set_title("Footprint: worst class + centroid path", fontsize=8)
    for ax in flat[n:]:
        ax.axis("off")
    handles = [LegendPatch(color=c, label=l) for c, l in zip(CLASS_COLORS, CLASS_LABELS)]
    handles += [Line2D([], [], marker="o", ls="", mfc="white", mec="k", label="monthly centroid"),
                Line2D([], [], marker="*", ls="", mfc="gold", mec="k", ms=11, label="event centroid")]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=7.5, frameon=False)
    shown = "" if len(tis_show) == len(tis) else f"  (showing {len(tis_show)} of {len(tis)} months)"
    fig.suptitle(f"Event {event_id}: {ce.t_start} → {ce.t_end}   D = {ce.D} mo · A = {ce.A_km2:,.0f} · S = {ce.S:,.1f} · I = {ce.I:.2f}{shown}", y=1.0, fontsize=10)
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    return fig


def plot_event_profile(res, event_id):
    """Monthly area and severity of one event (from the monthly track O2)."""
    te = res["track"][res["track"]["event_id"] == event_id].sort_values("t_index")
    fig, axs = plt.subplots(1, 3, figsize=(11, 2.8))
    x = np.arange(len(te))
    for ax, col, ttl, c in zip(axs, ["area_km2", "sev", "mean_abs_spei"],
                               ["Area in drought", "Severity Σ|SPEI|·A", "Mean |SPEI|"], ["#6baed6", "#e6550d", "#7f2704"]):
        ax.bar(x, te[col], color=c)
        ax.set_xticks(x); ax.set_xticklabels(te["month"], rotation=60, fontsize=7); ax.set_title(ttl)
    fig.suptitle(f"Event {event_id} – how it grew and faded", y=1.02)
    fig.tight_layout()
    return fig


def plot_event_3d(res, spei, lat, lon, times, event_id, max_points=5000):
    """Event as a 3D body (lon × lat × time), cells coloured by |SPEI|, with the centroid path."""
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    cube = res["label_cube"]
    t, y, x = np.nonzero(cube == event_id)
    if t.size > max_points:
        sel = np.random.default_rng(0).choice(t.size, max_points, replace=False)
        t, y, x = t[sel], y[sel], x[sel]
    fig = plt.figure(figsize=(6.5, 5))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(np.asarray(lon)[x], np.asarray(lat)[y], t + 1, c=np.abs(spei[t, y, x]), cmap="YlOrRd", s=14, alpha=0.7, vmin=1, vmax=3)
    te = res["track"][res["track"]["event_id"] == event_id].sort_values("t_index")
    ce = res["catalogue"].set_index("event_id").loc[event_id]
    ax.plot(te["lon_c"], te["lat_c"], te["t_index"], "-o", color="k", ms=3, lw=1.2, label="monthly centroids")
    ax.scatter([ce.lon_c], [ce.lat_c], [ce.t_c], marker="*", s=180, c="gold", edgecolor="k", label="3D centroid", zorder=10)
    ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.set_zlabel("month no.")
    fig.colorbar(sc, ax=ax, shrink=0.6, label="|SPEI|"); ax.legend(fontsize=7, loc="upper left")
    ax.set_title(f"Event {event_id} in 3D (lon × lat × time)")
    fig.tight_layout()
    return fig


def plot_event_summary(res, times, top_n=20):
    """Overview of all events: timeline of the strongest ones, severity vs duration, durations, events per year."""
    cat = res["catalogue"].copy()
    fig, axs = plt.subplots(2, 2, figsize=(12, 7.5))
    top = cat.sort_values("S", ascending=False).head(top_n).sort_values("t_start")
    ax = axs[0, 0]
    for k, r in enumerate(top.itertuples()):
        s, e = pd.Timestamp(r.t_start), pd.Timestamp(r.t_end) + pd.offsets.MonthEnd(0)
        ax.barh(k, (e - s).days + 1, left=mdates_num(s), color=plt.cm.YlOrRd(min(r.I - 1, 1.5) / 1.5), edgecolor="k", lw=0.4)
    ax.set_yticks(range(len(top))); ax.set_yticklabels([f"E{i}" for i in top["event_id"]], fontsize=7)
    ax.xaxis_date(); ax.set_title(f"Timeline of the {len(top)} most severe events (colour = intensity)")
    ax = axs[0, 1]
    sc = ax.scatter(cat["D"], cat["S"], s=15 + 5 * cat["A_pct"], c=cat["I"], cmap="YlOrRd", edgecolor="k", lw=0.4)
    ax.set_xlabel("duration D (months)"); ax.set_ylabel("severity S"); ax.set_yscale("log")
    fig.colorbar(sc, ax=ax, label="intensity I"); ax.set_title("Severity vs duration (size = % of basin)")
    ax = axs[1, 0]
    vc = cat["D"].value_counts().sort_index()
    ax.bar(vc.index, vc.values, color="#6baed6"); ax.set_xlabel("duration D (months)"); ax.set_ylabel("events"); ax.set_title("Duration of events")
    ax = axs[1, 1]
    yr = pd.to_datetime(cat["t_start"]).dt.year.value_counts().sort_index()
    ax.bar(yr.index, yr.values, color="#e6550d"); ax.set_xlabel("year of start"); ax.set_ylabel("events"); ax.set_title("Events started per year")
    fig.tight_layout()
    return fig


def mdates_num(ts):
    import matplotlib.dates as mdates
    return mdates.date2num(ts)


def plot_frequency_map(res, lat, lon, times):
    """Share of months each cell is part of a drought event, and the number of different events that hit it."""
    cube = res["label_cube"]
    inev = cube > 0
    lon_e, lat_e = _edges(lon), _edges(lat)
    n_ev = np.zeros(inev.shape[1:], int)
    for e in np.unique(cube[inev]):
        n_ev += (cube == e).any(0)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
    valid = np.isfinite(res["_spei_ref"]).any(0) if "_spei_ref" in res else np.ones(inev.shape[1:], bool)
    for ax, arr, ttl, cm in zip(axs, [100 * inev.mean(0), n_ev], ["% of months inside a drought event", "Number of different events"], ["YlOrRd", "PuRd"]):
        m = ax.pcolormesh(lon_e, lat_e, np.ma.masked_where(~valid, arr), cmap=cm)
        fig.colorbar(m, ax=ax, shrink=0.8); ax.set_title(ttl); ax.set_aspect(_aspect(res, lat))
    fig.tight_layout()
    return fig

### J. Output writers (O1–O5) and the main function `run_workflow`

In [ ]:
def write_outputs(res, out_dir, lat, lon, times, var_name="spei3", params=None, extra_log=None):
    """Write O1 (catalogue CSV), O2 (monthly track CSV), O3 (label-cube NetCDF), O5 (run log JSON) and the patch / link tables."""
    os.makedirs(out_dir, exist_ok=True)
    paths = {}
    cat = res["catalogue"].copy()
    for c in ["A_km2", "A_pct", "V", "S", "I", "lon_c", "lat_c", "t_c"]:
        cat[c] = cat[c].astype(float).round(4)
    paths["O1"] = os.path.join(out_dir, "O1_event_catalogue.csv"); cat.to_csv(paths["O1"], index=False)
    tr = res["track"].drop(columns=["t_index"]).copy()
    for c in ["area_km2", "sev", "mean_abs_spei", "lon_c", "lat_c"]:
        tr[c] = tr[c].astype(float).round(4)
    paths["O2"] = os.path.join(out_dir, "O2_monthly_track.csv"); tr.to_csv(paths["O2"], index=False)
    paths["O3"] = write_nc(os.path.join(out_dir, "O3_event_label_cube.nc"), "event_id", res["label_cube"], lat, lon, times,
                           dtype="int32", lat_name="lat", lon_name="lon",
                           attrs=dict(long_name="drought event id (0 = no event)", threshold=res["threshold"],
                                      source_variable=var_name))
    paths["patches"] = os.path.join(out_dir, "patches.csv"); res["patch_df"].to_csv(paths["patches"], index=False)
    paths["links"] = os.path.join(out_dir, "links.csv"); res["links_df"].to_csv(paths["links"], index=False)
    paths["months"] = os.path.join(out_dir, "month_stats.csv"); res["month_df"].to_csv(paths["months"], index=False)
    return paths


def make_run_log(res, data, params, qc_info, area_info, paths, runtime_s=None):
    """O5 - everything needed to repeat the run."""
    lat, lon, t = data["lat"], data["lon"], data["time"]
    ldf = res["links_df"]
    ver = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
               scipy=__import__("scipy").__version__, matplotlib=matplotlib_version(), xarray=getattr(xr, "__version__", None))
    return dict(
        created=str(pd.Timestamp.now().round("s")),
        input=dict(file=str(data.get("path")), variable=data.get("var"), n_time=len(t), n_lat=len(lat), n_lon=len(lon),
                   grid_step_deg=[float(np.median(np.diff(lat))) if len(lat) > 1 else None, float(np.median(np.diff(lon))) if len(lon) > 1 else None]),
        period=dict(start=f"{t[0]:%Y-%m}", end=f"{t[-1]:%Y-%m}", months=len(t)),
        study_area=dict(north=float(lat.max()), south=float(lat.min()), west=float(lon.min()), east=float(lon.max()),
                        n_cells_total=area_info["n_cells_total"], n_cells_valid=area_info["n_cells_valid"],
                        share_clipped_no_data=round(area_info["share_no_data"], 4)),
        A_tot_km2=round(res["A_tot"], 3), A_patch_km2=round(res["min_patch_area"], 3), A_overlap_km2=round(res["link_min_area"], 3),
        parameters=params, quality_control=qc_info,
        counts=dict(drought_cells=int(res["month_df"]["n_drought_cells"].sum()),
                    patches_kept=int(len(res["patch_df"])),
                    patches_dropped_small=int(res["month_df"]["n_dropped_small"].sum()),
                    single_cells_dropped=int(res["month_df"]["n_dropped_single"].sum()),
                    overlapping_pairs=int(len(ldf)), links_passed=int(ldf["linked"].sum()) if len(ldf) else 0,
                    events=int(len(res["catalogue"])),
                    events_dur_ge3=int(res["catalogue"]["dur_ge3"].sum()) if len(res["catalogue"]) else 0,
                    events_censored=int(res["catalogue"]["censored"].sum()) if len(res["catalogue"]) else 0),
        versions=ver, outputs={k: os.path.basename(v) if isinstance(v, str) else v for k, v in paths.items()},
        runtime_seconds=runtime_s)


def matplotlib_version():
    import matplotlib
    return matplotlib.__version__


def run_workflow(input_nc, out_dir, var=None, bbox=None, period=None,
                 threshold=-1.0, min_area_frac=0.05, link_frac=0.05, connectivity=8,
                 qc=False, alpha=0.05, unit_cell_area=False,
                 show=True, max_event_maps=None, show_max_maps=3, max_panels=12, make_3d=True, dpi=150):
    """Run the whole drought-event workflow and save + show all outputs.

    Parameters
    ----------
    input_nc      : NetCDF file with monthly SPEI (time, lat, lon)
    out_dir       : directory where everything is saved (created if missing)
    var           : SPEI variable name (None = auto-detect, e.g. 'spei3')
    bbox          : dict(north=, south=, west=, east=) study area, or None for the whole grid
    period        : ('YYYY-MM', 'YYYY-MM') inclusive, or None
    threshold     : drought cell if SPEI < threshold (default -1)
    min_area_frac : keep patch if area > this share of the basin (default 0.05)
    link_frac     : link patches if overlap > this share of the basin (default 0.05)
    connectivity  : 8 (sides + corners, default) or 4
    qc, alpha     : optional Shapiro-Wilk normality test; cells that fail are masked (not used for drought detection)
    unit_cell_area: True = every cell counts as 1 (toy examples); False = true km² per cell
    show          : display tables / figures in the notebook (figures are always saved)
    max_event_maps: limit the number of per-event PNG maps (None = one for every event; strongest events first)
    show_max_maps : how many event maps to display in the notebook
    Returns a dict with all results and the paths of the saved files.
    """
    import time as _time
    t_start = _time.time()
    os.makedirs(out_dir, exist_ok=True)
    fig_dir = os.path.join(out_dir, "figures")
    map_dir = os.path.join(out_dir, "O4_event_maps")
    os.makedirs(fig_dir, exist_ok=True); os.makedirs(map_dir, exist_ok=True)

    # ---- Step 0: read, subset, QC, areas -------------------------------------------------------------------
    print("Step 0 | reading and preparing data ...")
    data = subset_spei(read_spei_nc(input_nc, var), bbox, period)
    spei, lat, lon, times = data["spei"].copy(), data["lat"], data["lon"], data["time"]
    area = np.ones((len(lat), len(lon))) if unit_cell_area else cell_area_km2(lat, lon)
    area_info = study_area_stats(spei, area)          # A_tot from all cells with data (before QC)
    qc_info = dict(applied=bool(qc), test="Shapiro-Wilk", alpha=alpha if qc else None)
    paths, figs = {}, {}
    if qc:
        ok, pv = normality_qc_mask(spei, alpha)
        spei[:, ~ok] = np.nan
        qc_info.update(cells_tested=int(np.isfinite(pv).sum()), cells_failed=int((np.isfinite(pv) & ~ok).sum()))
        figs["qc"] = _finish(plot_qc(ok, pv, lat, lon, alpha), os.path.join(fig_dir, "qc_normality.png"), show, dpi)
    print(f"        {len(times)} months ({times[0]:%Y-%m} – {times[-1]:%Y-%m}), grid {len(lat)} x {len(lon)}, "
          f"A_tot = {area_info['A_tot']:,.1f} {'cells' if unit_cell_area else 'km²'}, no-data share = {100 * area_info['share_no_data']:.1f} %")

    # ---- Steps 1-3 + event numbers -------------------------------------------------------------------------
    print("Steps 1-3 | patches -> links -> events -> event numbers ...")
    res = detect_events(spei, lat, lon, times, area, threshold, min_area_frac, link_frac, connectivity, A_tot=area_info["A_tot"])
    res["unit_cell_area"] = unit_cell_area
    res["_spei_ref"] = spei
    cat = res["catalogue"]
    print(f"        patches kept: {len(res['patch_df'])} | overlapping pairs: {len(res['links_df'])} | "
          f"links passed: {int(res['links_df']['linked'].sum()) if len(res['links_df']) else 0} | events: {len(cat)}")

    # ---- Outputs O1, O2, O3 ---------------------------------------------------------------------------------
    print("Outputs | writing O1 catalogue, O2 monthly track, O3 label cube ...")
    params = dict(threshold=threshold, min_area_frac=min_area_frac, link_frac=link_frac, connectivity=connectivity,
                  qc=qc, alpha=alpha, unit_cell_area=unit_cell_area, bbox=bbox, period=period, var=data["var"])
    paths.update(write_outputs(res, out_dir, lat, lon, times, data["var"], params))
    if show:
        show_table(cat.round(3), "O1 · Event catalogue")
        show_table(res["track"].drop(columns=["t_index"]).round(3), "O2 · Monthly track")
        show_table(res["patch_df"], "Patches kept (Step 1)")
        show_table(res["links_df"], "Overlaps between consecutive months (Step 2) – linked = overlap > 5 % of basin")

    # ---- Figures --------------------------------------------------------------------------------------------
    print("Figures | overview, step pictures, event maps ...")
    figs["overview"] = _finish(plot_overview(res, spei, lat, lon, times, threshold), os.path.join(fig_dir, "input_overview.png"), show, dpi)
    if len(res["patch_df"]):
        first_months = list(res["patch_df"]["t_index"].drop_duplicates().head(4) - 1)
        figs["step1"] = _finish(plot_step1_patches(res, spei, lat, lon, times, months=first_months), os.path.join(fig_dir, "step1_patches.png"), show, dpi)
    if len(cat):
        figs["summary"] = _finish(plot_event_summary(res, times), os.path.join(fig_dir, "events_summary.png"), show, dpi)
        figs["frequency"] = _finish(plot_frequency_map(res, lat, lon, times), os.path.join(fig_dir, "event_frequency_map.png"), show, dpi)
        order = cat.sort_values("S", ascending=False)["event_id"].tolist()          # strongest first
        top = order[0]
        n_nodes = len(res["patch_df"])
        gid = None if n_nodes <= 60 else top
        figs["link_graph"] = _finish(plot_link_graph(res, times, event_id=gid), os.path.join(fig_dir, "link_graph.png"), show, dpi)
        figs["top_event_profile"] = _finish(plot_event_profile(res, top), os.path.join(fig_dir, f"event_{top:04d}_profile.png"), show, dpi)
        if make_3d:
            figs["top_event_3d"] = _finish(plot_event_3d(res, spei, lat, lon, times, top), os.path.join(fig_dir, f"event_{top:04d}_3d.png"), show, dpi)
        todo = order if max_event_maps is None else order[:max_event_maps]
        shown = set(order[:show_max_maps]) if show else set()
        map_paths = {}
        for e in todo:
            p = os.path.join(map_dir, f"event_{e:04d}.png")
            _finish(plot_event_map(res, spei, lat, lon, times, e, max_panels), p, show=(e in shown), dpi=dpi)
            map_paths[e] = p
        paths["O4"] = map_dir
        print(f"        saved {len(map_paths)} event map(s) in {map_dir}")
    else:
        print("        no events found with these parameters - nothing to map.")

    # ---- O5 run log ----------------------------------------------------------------------------------------
    paths["O5"] = os.path.join(out_dir, "O5_run_log.json")
    log = make_run_log(res, data, params, qc_info, area_info, paths, runtime_s=round(_time.time() - t_start, 2))
    with open(paths["O5"], "w") as f:
        json.dump(log, f, indent=2, default=_json_default)
    if show:
        print("\nO5 · Run log"); print(json.dumps(log, indent=2, default=_json_default)[:2500])
    print(f"\nDone in {_time.time() - t_start:.1f} s. Everything is in: {os.path.abspath(out_dir)}")
    for k, v in paths.items():
        print(f"  {k:9s} {v}")
    return dict(results=res, log=log, paths=paths, figures=figs, data=data)

## 3. Test 1 – the toy basin from the slides (20 × 10 cells, Jan–Apr)
The slides only show pictures of the toy maps, so the toy field below is **re-drawn by hand with the same structure**:
7 patches (A | B C D | E F | G), one split (A → B, C, D), one merge (B, C → E), two overlaps that are too small
(D–F = 4 cells), small patches (≤ 10 cells) and single cells that must be dropped, and a cell with SPEI = exactly −1.0
that must **not** count. Cell counts therefore differ slightly from the slide, but every rule is exercised.
Three checks follow: (a) the formulas reproduce the **numbers printed in the slides**, (b) the workflow result equals an
independent slow pure-Python implementation, (c) the designed structure is found.

In [ ]:
def make_toy_spei(path, seed=1):
    """Toy SPEI field (4 months x 10 rows x 20 columns) saved as NetCDF; lon = column x, lat = row y."""
    rng = np.random.default_rng(seed)
    T, Y, X = 4, 10, 20
    spei = rng.uniform(-0.9, 0.9, (T, Y, X))                      # background: never below -1

    def rect(x0, x1, y0, y1):
        m = np.zeros((Y, X), bool); m[y0:y1 + 1, x0:x1 + 1] = True; return m

    def pts(*xy):
        m = np.zeros((Y, X), bool)
        for x, y in xy:
            m[y, x] = True
        return m

    month_masks = [
        # Jan: A + (5-cell, 4-cell patch) + 3 single cells
        rect(0, 12, 2, 6) | rect(15, 17, 0, 0) | rect(15, 16, 1, 1) | rect(18, 19, 8, 9) | pts((15, 4), (17, 6), (19, 3)),
        # Feb: B, C, D + (6, 3) + 3 singles
        rect(0, 2, 1, 7) | rect(4, 6, 1, 7) | rect(8, 12, 4, 8) | rect(15, 17, 0, 1) | rect(16, 18, 8, 8) | pts((19, 3), (15, 5), (18, 5)),
        # Mar: E, F + (8, 4) + 2 singles
        rect(0, 6, 3, 8) | rect(9, 16, 8, 9) | rect(13, 16, 7, 7) | rect(15, 18, 2, 3) | rect(8, 11, 0, 0) | pts((19, 5), (12, 2)),
        # Apr: G + (6, 3) + 3 singles
        rect(12, 18, 7, 9) | rect(0, 2, 0, 1) | rect(8, 10, 3, 3) | pts((18, 0), (5, 5), (14, 2)),
    ]
    for t, m in enumerate(month_masks):
        spei[t][m] = -(1.1 + 1.2 * rng.random(m.sum()))             # all strictly below -1
    spei[0, 4, 13] = -1.0                                           # exactly -1.0 next to A: must NOT be drought
    times = pd.date_range("2001-01-01", periods=T, freq="MS")
    write_nc(path, "spei3", spei, np.arange(Y, dtype=float), np.arange(X, dtype=float), times)
    return path


def brute_force_events(spei, thr=-1.0, frac=0.05):
    """Independent, deliberately simple (slow) re-implementation with python sets - used only to cross-check."""
    T, Y, X = spei.shape
    total = Y * X
    patches = []
    for t in range(T):
        seen = set()
        for y in range(Y):
            for x in range(X):
                if spei[t, y, x] < thr and (y, x) not in seen:
                    comp, stack = {(y, x)}, [(y, x)]
                    seen.add((y, x))
                    while stack:
                        cy, cx = stack.pop()
                        for dy in (-1, 0, 1):
                            for dx in (-1, 0, 1):
                                n = (cy + dy, cx + dx)
                                if 0 <= n[0] < Y and 0 <= n[1] < X and n not in seen and spei[t, n[0], n[1]] < thr:
                                    seen.add(n); comp.add(n); stack.append(n)
                    if len(comp) > frac * total:
                        patches.append((t, comp))
    parent = list(range(len(patches)))

    def find(i):
        while parent[i] != i:
            i = parent[i]
        return i

    for i, (ti, ci) in enumerate(patches):
        for j, (tj, cj) in enumerate(patches):
            if tj == ti + 1 and len(ci & cj) > frac * total:
                parent[max(find(i), find(j))] = min(find(i), find(j))
    groups = {}
    for i in range(len(patches)):
        groups.setdefault(find(i), []).append(i)
    events = []
    for r in sorted(groups):
        mem = groups[r]
        foot = set().union(*[patches[i][1] for i in mem])
        V = S = sx = sy = st = 0.0
        for i in mem:
            t, comp = patches[i]
            for (y, x) in comp:
                w = abs(spei[t, y, x]); V += 1; S += w; sx += w * x; sy += w * y; st += w * (t + 1)
        ts = [patches[i][0] for i in mem]
        events.append(dict(footprint=len(foot), V=V, S=S, I=S / V, lon_c=sx / S, lat_c=sy / S, t_c=st / S,
                           D=max(ts) - min(ts) + 1, n_patches=len(mem)))
    return events


def check_slide_formulas():
    """Feed the monthly tables printed in the slides (slides 8, 11, 12) into `summarise_events` and compare with the slides."""
    times = pd.date_range("2001-01-01", periods=4, freq="MS")
    rows = [  # event, t, patches, cells, sev, x, y
        (1, 1, "A", 65, 102.6, 7.4, 5.0), (1, 2, "B+C+D", 67, 90.4, 7.7, 5.0), (1, 3, "E", 44, 60.4, 8.5, 2.6),
        (2, 3, "F", 32, 45.9, 12.9, 8.1), (2, 4, "G", 42, 61.4, 15.8, 7.3)]
    tr = pd.DataFrame(rows, columns=["event_id", "t_index", "patch_ids", "area_km2", "sev", "lon_c", "lat_c"])
    tr["month"] = [f"{times[t - 1]:%Y-%m}" for t in tr["t_index"]]
    tr["n_cells"] = tr["area_km2"]; tr["mean_abs_spei"] = tr["sev"] / tr["area_km2"]
    cat = summarise_events(tr, pd.Series({1: 99.0, 2: 56.0}), pd.Series({1: 5, 2: 2}), 200.0, times).set_index("event_id")
    exp = {1: dict(D=3, V=176, S=253.4, I=1.44, lon_c=7.77, lat_c=4.43, t_c=1.83, A_km2=99),
           2: dict(D=2, V=74, S=107.3, I=1.45, lon_c=14.56, lat_c=7.64, t_c=3.57, A_km2=56)}
    for e, d in exp.items():
        for k, v in d.items():
            assert abs(cat.loc[e, k] - v) < 0.006, (e, k, cat.loc[e, k], v)
    print("OK - the formulas reproduce the slide numbers (D, area, area x time, S, I, centroid) for both events.")
    print("(Event 2 x-centroid is 14.56 from the rounded monthly values; the slide's 14.5 came from unrounded cell values.)")
    return cat.reset_index()


slide_check = check_slide_formulas()
display(slide_check.round(3))

In [ ]:
WORK = "/content/drought_workflow" if os.path.isdir("/content") else os.path.abspath("drought_workflow")
os.makedirs(WORK, exist_ok=True)

toy_nc = make_toy_spei(os.path.join(WORK, "toy_spei3.nc"))
toy = run_workflow(toy_nc, os.path.join(WORK, "toy_output"), unit_cell_area=True, show=True, make_3d=True)

### Toy checks
Compare with the independent implementation and with the structure that was drawn on purpose.

In [ ]:
res = toy["results"]
spei_toy = toy["data"]["spei"]
bf = brute_force_events(spei_toy)
cat = res["catalogue"]
assert len(bf) == len(cat) == 2, (len(bf), len(cat))
for b, (_, r) in zip(bf, cat.iterrows()):
    assert b["footprint"] == r.A_km2 and b["D"] == r.D and b["n_patches"] == r.n_patches
    for k, c in [("V", "V"), ("S", "S"), ("I", "I"), ("lon_c", "lon_c"), ("lat_c", "lat_c"), ("t_c", "t_c")]:
        assert abs(b[k] - r[c]) < 1e-9, (k, b[k], r[c])
print("OK 1/3  workflow == independent brute-force implementation (footprint, V, S, I, centroid, D, patches)")

pdf, ldf = res["patch_df"], res["links_df"]
assert list(pdf["name"]) == list("ABCDEFG") and list(pdf["n_cells"]) == [65, 21, 21, 25, 42, 20, 21]
assert (res["month_df"]["n_dropped_small"] == 2).all() and list(res["month_df"]["n_dropped_single"]) == [3, 3, 2, 3]
passed = sorted(zip(ldf[ldf.linked].name_p, ldf[ldf.linked].name_q))
assert passed == [("A", "B"), ("A", "C"), ("A", "D"), ("B", "E"), ("C", "E"), ("F", "G")], passed
failed = ldf[~ldf.linked]
assert list(zip(failed.name_p, failed.name_q, failed.overlap_cells)) == [("D", "F", 4)]
print("OK 2/3  7 patches A-G, small patches/single cells dropped, the -1.0 cell ignored, 6 links, D-F (4 cells) not linked")

assert list(cat["D"]) == [3, 2] and list(cat["A_km2"]) == [95, 27] and list(cat["V"]) == [174, 41]
ev = res["label_cube"]
assert set(np.unique(ev[:3][ev[:3] > 0])) >= {1} and (ev[3] == 2).sum() == 21
nc_back = read_spei_nc(os.path.join(toy["paths"]["O3"]), var="event_id")
assert np.array_equal(np.nan_to_num(nc_back["spei"]).astype(int), ev)
print("OK 3/3  two events (Jan-Mar, 95 cells | Mar-Apr, 27 cells), split + merge kept in one event, O3 NetCDF round-trips")

## 4. Test 2 – dummy NetCDF data on the sample grid (N 50, S 45, W 1°W, E 6°E; 0.25°; 1980–2023)
Generate a dummy SPEI-3 file, then call `run_workflow` exactly as you would on real data.

In [ ]:
dummy_nc = make_dummy_spei(os.path.join(WORK, "dummy_spei3.nc"), seed=42)
d = read_spei_nc(dummy_nc)
print("dummy file:", dummy_nc, "| variable:", d["var"], "| shape (time, lat, lon):", d["spei"].shape,
      f"| {d['time'][0]:%Y-%m} → {d['time'][-1]:%Y-%m} | SPEI range {np.nanmin(d['spei']):.2f} … {np.nanmax(d['spei']):.2f}")

In [ ]:
out = run_workflow(
    input_nc=dummy_nc,
    out_dir=os.path.join(WORK, "dummy_output"),
    var="spei3",
    bbox=dict(north=50, south=45, west=-1, east=6),
    period=("1980-01", "2023-12"),
    threshold=-1.0, min_area_frac=0.05, link_frac=0.05, connectivity=8,
    qc=False, show=True,
    max_event_maps=25,        # demo: only the 25 strongest events as PNG (None = one PNG for every event)
    show_max_maps=3)

### Optional quality criterion (normality test of SPEI, α = 0.05)
Cells whose SPEI series fails the Shapiro-Wilk test are masked and cannot be drought cells. (The planted droughts make
the dummy series non-normal, so many cells fail here – on a real, well-behaved SPEI file the pass rate is typically high.)

In [ ]:
out_qc = run_workflow(dummy_nc, os.path.join(WORK, "dummy_output_with_QC"), var="spei3", qc=True, alpha=0.05, show=False, max_event_maps=5)
print("events without QC:", len(out["results"]["catalogue"]), "| events with QC:", len(out_qc["results"]["catalogue"]))
q = out_qc["log"]["quality_control"]; print("QC:", q)
from matplotlib.image import imread
fig = plt.figure(figsize=(9, 3.4)); plt.imshow(imread(out_qc["figures"]["qc"])); plt.axis("off"); plt.show()

### Check the dummy-run files

In [ ]:
assert len(out["results"]["catalogue"]) > 0
cube_nc = read_spei_nc(out["paths"]["O3"], var="event_id")
assert np.array_equal(np.nan_to_num(cube_nc["spei"]).astype(int), out["results"]["label_cube"])
n_png = len([f for f in os.listdir(out["paths"]["O4"]) if f.endswith(".png")])
assert n_png == min(25, len(out["results"]["catalogue"]))
for k in ["O1", "O2", "O3", "O5"]:
    assert os.path.getsize(out["paths"][k]) > 0
print(f"OK - O1/O2/O3/O5 written, O3 round-trips, {n_png} event PNGs (strongest 25; use max_event_maps=None for all).")
print("\nFiles written:")
for root, _, files in os.walk(os.path.join(WORK, "dummy_output")):
    if "O4_event_maps" in root and root != os.path.join(WORK, "dummy_output", "O4_event_maps"):
        continue
    for fn in sorted(files)[:8] if "O4_event_maps" in root else sorted(files):
        print("  ", os.path.relpath(os.path.join(root, fn), WORK))
    if "O4_event_maps" in root and len(files) > 8:
        print(f"   ... and {len(files) - 8} more PNGs")

## 5. Run it on your own data
1. Put the ERA5-derived SPEI-3 NetCDF in Google Drive (or upload it with the Colab file panel).
2. Edit the paths / parameters below and run the cell. Everything (CSV, NetCDF, PNG, JSON) is saved in `OUT_DIR`.

`run_workflow` finds the time / latitude / longitude coordinates and the `spei3` variable automatically; use
`var="spei3"` to force it. For a larger region or the whole 1980–present record only the arguments change.

In [ ]:
RUN_ON_MY_DATA = False          # set True after editing the paths below

if RUN_ON_MY_DATA:
    # from google.colab import drive; drive.mount("/content/drive")
    INPUT_NC = "/content/drive/MyDrive/spei3_era5.nc"
    OUT_DIR = "/content/drive/MyDrive/drought_output"
    my_run = run_workflow(
        input_nc=INPUT_NC, out_dir=OUT_DIR, var=None,
        bbox=dict(north=50, south=45, west=-1, east=6),     # or None for the full grid
        period=("1980-01", "2023-12"),                        # or None
        threshold=-1.0, min_area_frac=0.05, link_frac=0.05, connectivity=8,
        qc=False, alpha=0.05, show=True,
        max_event_maps=None)                                 # e.g. 50 to save only the 50 strongest events as PNG

In [ ]:
# Optional: zip an output folder and download it from Colab
def zip_and_download(folder):
    z = shutil.make_archive(folder.rstrip("/"), "zip", folder)
    try:
        from google.colab import files
        files.download(z)
    except ImportError:
        print("zip created:", z)

# zip_and_download(os.path.join(WORK, "dummy_output"))